# spaCy + XGBoost Research (Advanced Features)

This notebook implements advanced feature extraction using spaCy and trains an XGBoost classifier. 

Key enhancements include:
- Enhanced negation logic (scope overlap via lemmas)
- POS-filtered overlaps
- Extended WordNet relationships (multilingual)
- Refined graph kernels (WL with POS and attributes)
- Soft lexical alignment (cosine similarity)
- Model optimizations (SVD, Optuna, early stopping, ensemble)

In [ ]:
import json
import gc
from pathlib import Path

import joblib
import mlflow
import optuna
import networkx as nx
import nltk
import numpy as np
import pandas as pd
import spacy
import spacy.cli
import xgboost as xgb
from nltk.corpus import wordnet as wn
from sklearn.metrics import accuracy_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split

from helpers.data import load_processed_data, load_sample_submission
from helpers.metrics import compute_all_metrics
from helpers.mlflow import log_common_context, log_metrics, log_model_artifacts, start_notebook_run
from helpers.submission import build_submission, save_submission

# Download required NLTK data
nltk.download('wordnet')
nltk.download('omw-1.4')
import matplotlib.pyplot as plt
from sklearn.decomposition import IncrementalPCA
from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler

# Mapping for multilingual support
LANG_MAP = {
    'en': {'model': 'en_core_web_lg', 'wn': 'eng'},
    'fr': {'model': 'fr_core_news_lg', 'wn': 'fra'},
    'ar': {'model': 'xx_sent_ud_sm', 'wn': 'arb'},
    'sw': {'model': 'xx_sent_ud_sm', 'wn': None},
    'vi': {'model': 'xx_sent_ud_sm', 'wn': None},
    'ur': {'model': 'xx_sent_ud_sm', 'wn': None},
    'zh': {'model': 'zh_core_web_lg', 'wn': 'cmn'},
    'el': {'model': 'el_core_news_lg', 'wn': 'ell'},
    'th': {'model': 'xx_sent_ud_sm', 'wn': 'tha'},
    'hi': {'model': 'xx_sent_ud_sm', 'wn': 'hin'},
    'tr': {'model': 'xx_sent_ud_sm', 'wn': 'tur'},
    'de': {'model': 'de_core_news_lg', 'wn': 'deu'},
    'es': {'model': 'es_core_news_lg', 'wn': 'spa'},
    'ru': {'model': 'ru_core_news_lg', 'wn': 'rus'},
    'bg': {'model': 'xx_sent_ud_sm', 'wn': 'bul'}
}

## Load Data

Loading the DVC-processed splits. If these are not available, the notebook will fail as it relies on the pre-processed data.

In [ ]:
PROCESSED_DIR = Path('../data/processed')
RAW_DIR = Path('../data/raw')

if (PROCESSED_DIR / 'train_split.csv').exists():
    train_df, val_df, test_df = load_processed_data(PROCESSED_DIR)
    sample_submission = load_sample_submission(RAW_DIR / 'sample_submission.csv')
    data_source = 'dvc_processed_split'
else:
    raise FileNotFoundError('Could not find DVC processed splits. Please run the data preparation scripts first.')

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Loaded {len(train_df)} training samples, {len(val_df)} validation samples.")

## Feature Extractor

The `extract_features` function uses spaCy to compute semantic similarity, lexical overlap (Jaccard and Overlap Coefficient), syntactic dependency overlap, and negation polarity mismatches.

In [ ]:
# Load spaCy model
try:
    nlp = spacy.load('en_core_web_lg')
except OSError:
    print('Downloading en_core_web_lg model...')
    spacy.cli.download('en_core_web_lg')
    nlp = spacy.load('en_core_web_lg')

def get_antonyms(word, lang='eng'):
    if not lang: return set()
    antonyms = set()
    try:
        for syn in wn.synsets(word, lang=lang):
            for l in syn.lemmas(lang=lang):
                if l.antonyms(): antonyms.add(l.antonyms()[0].name())
    except: pass
    return antonyms

def get_hypernyms(word, lang='eng'):
    if not lang: return set()
    hypernyms = set()
    try:
        for syn in wn.synsets(word, lang=lang):
            for hyper in syn.hypernyms():
                for l in hyper.lemmas(lang=lang): hypernyms.add(l.name())
    except: pass
    return hypernyms

def get_hyponyms(word, lang='eng'):
    if not lang: return set()
    hyponyms = set()
    try:
        for syn in wn.synsets(word, lang=lang):
            for hypo in syn.hyponyms():
                for l in hypo.lemmas(lang=lang): hyponyms.add(l.name())
    except: pass
    return hyponyms

IMPLICIT_NEGATION_EN = {'fail', 'deny', 'refuse', 'lack', 'without', 'stop', 'prevent', 'reject', 'miss', 'avoid'}

TARGET_VEC_DIM = 300
N_VEC_TOTAL = 2 + 2 * TARGET_VEC_DIM # indicator(1) + similarity(1) + diff(300) + prod(300)

def get_safe_vector(doc, target_dim=300):
    res = np.zeros(target_dim)
    if hasattr(doc, 'vector') and doc.has_vector:
        v = doc.vector
        curr_dim = v.shape[0]
        res[:min(curr_dim, target_dim)] = v[:min(curr_dim, target_dim)]
    return res

def get_avg_v(tokens, target_dim=300):
    if not tokens: return np.zeros(target_dim)
    vectors = [t.vector for t in tokens if t.has_vector]
    if not vectors: return np.zeros(target_dim)
    v_avg = np.mean(vectors, axis=0)
    res = np.zeros(target_dim)
    res[:min(v_avg.shape[0], target_dim)] = v_avg[:min(v_avg.shape[0], target_dim)]
    return res

def get_wl_hash(token, lang='en'):
    # Advanced Weisfeiler-Lehman: lemma + pos + attributes + sorted labels of neighbors
    neighs = sorted([c.dep_ for c in token.children])
    attrs = []
    if token.dep_ == 'neg' or (lang == 'en' and token.lemma_.lower() in IMPLICIT_NEGATION_EN): attrs.append('NEG')
    if token.ent_type_: attrs.append('NE')
    if token.like_num: attrs.append('NUM')
    attr_str = '|'.join(attrs)
    return f'{token.lemma_}_{token.pos_}_{token.dep_}_{attr_str}_' + '_'.join(neighs)

def compute_soft_alignment(doc1, doc2):
    if not doc1 or not doc2: return 0.0
    sims = []
    for t2 in doc2:
        if not t2.is_alpha or t2.is_stop: continue
        max_sim = 0.0
        for t1 in doc1:
            if not t1.is_alpha or t1.is_stop: continue
            if t1.has_vector and t2.has_vector:
                sim = t1.similarity(t2)
                if sim > max_sim: max_sim = sim
        sims.append(max_sim)
    return np.mean(sims) if sims else 0.0

def get_negation_scope_lemmas(doc, lang='en'):
    neg_tokens = [t for t in doc if t.dep_ == 'neg' or (lang == 'en' and t.lemma_.lower() in IMPLICIT_NEGATION_EN)]
    scope_lemmas = set()
    for t in neg_tokens:
        scope_lemmas.add(t.head.lemma_.lower())
        for child in t.head.children:
            scope_lemmas.add(child.lemma_.lower())
    return scope_lemmas

def compute_structural_alignment(doc_p, doc_h):
    if not doc_p or not doc_h: return [0.0, 0.0, 0.0, 0.0]
    if not doc_p.has_vector or not doc_h.has_vector: return [0.0, 0.0, 0.0, 0.0]
    
    syn_sims, lex_sims, sem_sims, deep_sims = [], [], [], []
    content_pos = {'NOUN', 'VERB', 'ADJ', 'ADV'}
    
    p_tokens_by_pos = {}
    for t in doc_p:
        if t.pos_ in content_pos and t.has_vector:
            p_tokens_by_pos.setdefault(t.pos_, []).append(t)
            
    def cos_sim(v1, v2):
        if np.all(v1 == 0) or np.all(v2 == 0): return 0.0
        return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-8)
            
    for t_h in doc_h:
        if t_h.pos_ not in content_pos or not t_h.has_vector: continue
        candidates = p_tokens_by_pos.get(t_h.pos_, [])
        if not candidates: continue
            
        best_t_p = None
        max_sim = -1.0
        for t_p in candidates:
            sim = t_h.similarity(t_p)
            if sim > max_sim: 
                max_sim = sim
                best_t_p = t_p
                
        if best_t_p is not None:
            deps_h = {c.dep_ for c in t_h.children}
            deps_p = {c.dep_ for c in best_t_p.children}
            syn_sims.append(len(deps_h.intersection(deps_p)) / len(deps_h.union(deps_p)) if deps_h.union(deps_p) else 1.0)
            
            lems_h = {c.lemma_.lower() for c in t_h.children if c.is_alpha}
            lems_p = {c.lemma_.lower() for c in best_t_p.children if c.is_alpha}
            lex_sims.append(len(lems_h.intersection(lems_p)) / len(lems_h.union(lems_p)) if lems_h.union(lems_p) else 1.0)
            
            sem_sims.append(cos_sim(get_avg_v(list(t_h.children)), get_avg_v(list(best_t_p.children))))
            deep_sims.append(cos_sim(get_avg_v(list(t_h.subtree)), get_avg_v(list(best_t_p.subtree))))
            
    return [
        np.mean(syn_sims) if syn_sims else 0.0,
        np.mean(lex_sims) if lex_sims else 0.0,
        np.mean(sem_sims) if sem_sims else 0.0,
        np.mean(deep_sims) if deep_sims else 0.0
    ]

def compute_safe_similarity(doc1, doc2):
    if doc1.has_vector and doc2.has_vector and doc1.vector.shape == doc2.vector.shape:
        return doc1.similarity(doc2)
    return 0.0

def extract_features(nlp, premise_texts, hypothesis_texts, lang_abv='en'):
    if not premise_texts:
        return np.empty((0, N_VEC_TOTAL + 19))
    
    wn_lang = LANG_MAP.get(lang_abv, {}).get('wn')
    features = []
    
    # Disable n_process to ensure stability in notebook environments
    premises_parsed = list(nlp.pipe(premise_texts, n_process=-1, batch_size=256))
    hypotheses_parsed = list(nlp.pipe(hypothesis_texts, n_process=-1, batch_size=256))
    
    for doc_p, doc_h in zip(premises_parsed, hypotheses_parsed):
        row_vec = [] 
        row_logical = [] 
        
        # --- 1. VECTOR BLOCK (Safe padding/truncating to 300) ---
        # Indicator: do we have vectors in this model?
        has_v = 1.0 if doc_p.has_vector else 0.0
        row_vec.append(has_v)
        row_vec.append(compute_safe_similarity(doc_p, doc_h))
        
        # get_safe_vector handles target_dim consistency automatically
        v_p = get_safe_vector(doc_p, target_dim=TARGET_VEC_DIM)
        v_h = get_safe_vector(doc_h, target_dim=TARGET_VEC_DIM)
        
        row_vec.extend(np.abs(v_p - v_h).tolist())
        row_vec.extend((v_p * v_h).tolist())
        
        # --- 2. LOGICAL BLOCK (Symmetric & Safe) ---
        wl_p = {get_wl_hash(t, lang=lang_abv) for t in doc_p}
        wl_h = {get_wl_hash(t, lang=lang_abv) for t in doc_h}
        # Jaccard for symmetry
        wl_sim = len(wl_p.intersection(wl_h)) / len(wl_p.union(wl_h)) if wl_p.union(wl_h) else 1.0
        row_logical.append(wl_sim)
        
        lemmas_p = {t.lemma_.lower() for t in doc_p if not t.is_stop and t.is_alpha}
        lemmas_h = {t.lemma_.lower() for t in doc_h if not t.is_stop and t.is_alpha}
        lex_overlap = len(lemmas_p.intersection(lemmas_h)) / len(lemmas_p.union(lemmas_h)) if lemmas_p.union(lemmas_h) else 1.0
        row_logical.append(lex_overlap)
        
        # Negation scope (Lemmas based)
        scope_p = get_negation_scope_lemmas(doc_p, lang=lang_abv)
        scope_h = get_negation_scope_lemmas(doc_h, lang=lang_abv)
        # Symmetric scope overlap
        if scope_p or scope_h:
            scope_overlap = len(scope_p.intersection(scope_h)) / len(scope_p.union(scope_h)) if scope_p.union(scope_h) else 1.0
        else:
            scope_overlap = 0.0
        row_logical.append(scope_overlap)
        
        # ... (rest of logic: antonyms, hypernyms, etc., using wn_lang) ...
        
        # Path-grams (length 2) - Jaccard
        pg_p = {(t.head.lemma_, t.dep_, t.lemma_) for t in doc_p if t.head != t}
        pg_h = {(t.head.lemma_, t.dep_, t.lemma_) for t in doc_h if t.head != t}
        pg_sim = len(pg_p.intersection(pg_h)) / len(pg_p.union(pg_h)) if pg_p.union(pg_h) else 1.0
        row_logical.append(pg_sim)
        
        # POS-filtered lexical overlap (content words) - Jaccard
        content_pos = {'NOUN', 'VERB', 'ADJ', 'ADV'}
        content_p = {t.lemma_.lower() for t in doc_p if t.pos_ in content_pos}
        content_h = {t.lemma_.lower() for t in doc_h if t.pos_ in content_pos}
        pos_overlap = len(content_p.intersection(content_h)) / len(content_p.union(content_h)) if content_p.union(content_h) else 1.0
        row_logical.append(pos_overlap)

        # Soft lexical alignment
        soft_align = compute_soft_alignment(doc_p, doc_h)
        row_logical.append(soft_align)
        
        # Structural soft alignment (4 features)
        struct_align = compute_structural_alignment(doc_p, doc_h)
        row_logical.extend(struct_align)
        
        antonym_match = 0
        if wn_lang:
            for lh in lemmas_h:
                if lemmas_p.intersection(get_antonyms(lh, lang=wn_lang)): antonym_match = 1; break
        row_logical.append(antonym_match)
        
        hyper_match = 0
        if wn_lang:
            for lh in lemmas_h:
                if lemmas_p.intersection(get_hypernyms(lh, lang=wn_lang)): hyper_match = 1; break
        row_logical.append(hyper_match)
        
        hypo_match = 0
        if wn_lang:
            for lh in lemmas_h:
                if lemmas_p.intersection(get_hyponyms(lh, lang=wn_lang)): hypo_match = 1; break
        row_logical.append(hypo_match)
        
        nums_p = {t.text for t in doc_p if t.ent_type_ in ('CARDINAL', 'DATE') or t.like_num}
        nums_h = {t.text for t in doc_h if t.ent_type_ in ('CARDINAL', 'DATE') or t.like_num}
        row_logical.append(1 if nums_h.difference(nums_p) else 0)
        
        neg_p = any(t.dep_ == 'neg' for t in doc_p)
        neg_h = any(t.dep_ == 'neg' for t in doc_h)
        neg_mismatch = 1 if neg_p != neg_h else 0
        row_logical.append(neg_mismatch)
        
        len_ratio = len(doc_p) / len(doc_h) if len(doc_h) > 0 else 0
        row_logical.append(len_ratio)
        
        # Interaction Features
        row_logical.append(lex_overlap * neg_mismatch)
        row_logical.append(lex_overlap * antonym_match)
        row_logical.append(lex_overlap * wl_sim)
        
        features.append(row_vec + row_logical)
    return np.array(features)

LOGICAL_FEATURE_NAMES = [
    'wl_sim', 'lex_overlap', 'scope_overlap', 'pg_sim', 'pos_overlap', 
    'soft_align', 'struct_syn_sim', 'struct_lex_sim', 'struct_sem_sim', 'struct_deep_sim',
    'antonym_match', 'hyper_match', 'hypo_match', 'num_conflict', 'neg_mismatch', 
    'len_ratio', 'overlap_neg_inter', 'antonym_overlap_inter', 'wl_overlap_inter'
]


## Multilingual Data Preparation

Processing all 15 languages by loading and unloading models to manage memory.

In [ ]:
from sklearn.decomposition import TruncatedSVD


class LanguageSpecificTransformer:
    def __init__(self, n_components=30):
        self.scaler = StandardScaler()
        self.svd = TruncatedSVD(n_components=n_components, random_state=42)
        self.n_components = n_components
        
    def fit(self, X_vecs):
        X_scaled = self.scaler.fit_transform(X_vecs)
        self.svd.fit(X_scaled)
        return self
        
    def transform(self, X_vecs, X_logical):
        X_scaled = self.scaler.transform(X_vecs)
        X_pca = self.svd.transform(X_scaled)
        return np.hstack([X_logical, X_pca])

def process_by_language(df, transformers=None, is_training=True):
    all_features_final = []
    all_indices = []
    new_transformers = {} if transformers is None else transformers
    
    for lang_abv, group in df.groupby('lang_abv'):
        print(f"Processing language: {lang_abv} ({len(group)} samples)")
        model_cfg = LANG_MAP.get(lang_abv, {'model': 'xx_sent_ud_sm'})
        model_name = model_cfg['model']
        
        try:
            nlp_local = spacy.load(model_name)
        except OSError:
            print(f"Downloading {model_name}...")
            spacy.cli.download(model_name)
            nlp_local = spacy.load(model_name)

        print("Extracting features...")
        raw_features = extract_features(nlp_local, group['premise'].tolist(), group['hypothesis'].tolist(), lang_abv=lang_abv)
        
        X_vecs = raw_features[:, :N_VEC_TOTAL]
        X_logical = raw_features[:, N_VEC_TOTAL:]
        
        if is_training:
            print("Training model...")
            transformer = LanguageSpecificTransformer(n_components=30)
            transformer.fit(X_vecs)
            new_transformers[lang_abv] = transformer
        
        transformer = new_transformers.get(lang_abv)
        print("Transforming features...")
        if transformer:
            final_features = transformer.transform(X_vecs, X_logical)
        else:
            # Fallback
            final_features = np.hstack([X_logical, np.zeros((len(X_logical), 30))])
            
        all_features_final.append(final_features)
        all_indices.extend(group.index.tolist())
        
        del nlp_local
        gc.collect()
        
    return np.vstack(all_features_final), all_indices, new_transformers

print("Extracting and transforming features for training set...")
X_train, train_idxs, lang_transformers = process_by_language(train_df, is_training=True)
y_train = train_df.loc[train_idxs, 'label'].values

print("\nExtracting and transforming features for validation set...")
X_val, val_idxs, _ = process_by_language(val_df, transformers=lang_transformers, is_training=False)
y_val = val_df.loc[val_idxs, 'label'].values

# Update dataframes to match the feature order
train_df_ordered = train_df.loc[train_idxs].reset_index(drop=True)
val_df_ordered = val_df.loc[val_idxs].reset_index(drop=True)

In [ ]:
X_train_final = X_train
X_val_final = X_val
print(f"Using Language-Specific SVDs. Final feature count: {X_train_final.shape[1]}")

## Hyperparameter Search with Optuna

Using Optuna to find the best XGBoost parameters for the global multilingual dataset.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

def objective(trial):
    param = {
        'objective': 'multi:softprob',
        'num_class': 3,
        'tree_method': 'hist',
        'random_state': 42,
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 1e-8, 5.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
        'max_delta_step': trial.suggest_int('max_delta_step', 0, 5)
    }
    
    model = xgb.XGBClassifier(**param)
    score = cross_val_score(model, X_train_final, y_train, cv=3, scoring='accuracy').mean()
    return score

print("Starting Optuna Optimization...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20) # 20 trials for balance between quality and speed

print(f"Best trial Score: {study.best_value:.4f}")
print(f"Best Parameters: {study.best_params}")

best_params = study.best_params
best_params.update({
    'objective': 'multi:softprob',
    'num_class': 3,
    'tree_method': 'hist',
    'random_state': 42,
    'early_stopping_rounds': 20
})


## Model Training and MLflow Logging

We train a global XGBoost model on all languages, and then separate models for each of the 15 languages. All runs are logged to MLflow.

In [ ]:
def train_and_log_model(X_train, y_train, X_val, y_val, val_df_subset, run_name, tags, params=None):
    print(f"Training model: {run_name}...")
    if params is None: params = best_params
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    preds = model.predict(X_val)
    metrics = compute_all_metrics(val_df_subset, preds)
    
    with start_notebook_run(run_name, tags=tags):
        mlflow.log_params(params)
        log_metrics(metrics)
        log_common_context('../configs/data_split.yaml', '../data/processed/split_metadata.json')
        model_path = Path('../submissions') / f'{run_name}_model.joblib'
        joblib.dump(model, model_path)
        mlflow.log_artifact(str(model_path), artifact_path='model')
    return model, metrics

class LanguageSpecificEnsemble:
    def __init__(self, models_dict, transformers_dict):
        self.models = models_dict
        self.transformers = transformers_dict
    def predict(self, X_transformed, lang_abvs):
        preds = np.zeros(len(X_transformed), dtype=int)
        for lang, model in self.models.items():
            mask = (lang_abvs == lang)
            if mask.any():
                preds[mask] = model.predict(X_transformed[mask])
        return preds

# 1. Train Global Model
global_model, global_metrics = train_and_log_model(
    X_train_final, y_train, X_val_final, y_val, val_df_ordered, 
    "xgboost_spacy_global", 
    {'model_family': 'xgboost', 'scope': 'global', 'features': 'spacy_advanced_svd'}
)
print(f"Global Accuracy: {global_metrics['accuracy']:.4f}")
best_preds = global_model.predict(X_val_final)

# 2. Train Language-Specific Models (Logged as one Ensemble)
print("\nTraining Per-Language Ensemble...")
ensemble_models = {}
for lang_abv in val_df_ordered['lang_abv'].unique():
    train_mask = (train_df_ordered['lang_abv'] == lang_abv)
    val_mask = (val_df_ordered['lang_abv'] == lang_abv)
    
    if train_mask.sum() < 20:
        print(f"Skipping {lang_abv} ({train_mask.sum()} samples)")
        continue
        
    model = xgb.XGBClassifier(**best_params)
    # Safety check for empty validation set to avoid crash
    eval_set = [(X_val_final[val_mask], y_val[val_mask])] if val_mask.sum() > 0 else None
    
    model.fit(X_train_final[train_mask], y_train[train_mask], 
              eval_set=eval_set, verbose=False)
    ensemble_models[lang_abv] = model

ensemble = LanguageSpecificEnsemble(ensemble_models, lang_transformers)
ensemble_preds = ensemble.predict(X_val_final, val_df_ordered['lang_abv'].values)
ensemble_metrics = compute_all_metrics(val_df_ordered, ensemble_preds)

with start_notebook_run("xgboost_spacy_per_language_ensemble", 
                        tags={'model_family': 'xgboost', 'scope': 'per_language_ensemble', 'features': 'spacy_advanced_svd'}):
    mlflow.log_params(best_params)
    log_metrics(ensemble_metrics)
    # Log individual language accuracies as well
    for lang, model in ensemble_models.items():
        mask = (val_df_ordered['lang_abv'] == lang)
        if mask.any():
            lang_acc = accuracy_score(y_val[mask], model.predict(X_val_final[mask]))
            mlflow.log_metric(f"acc_{lang}", lang_acc)
    
    log_common_context('../configs/data_split.yaml', '../data/processed/split_metadata.json')
    ensemble_path = Path('../submissions') / 'xgboost_spacy_per_language_ensemble.joblib'
    joblib.dump(ensmble, ensemble_path)
    mlflow.log_artifact(str(ensemble_path), artifact_path='model')

print(f"Ensemble Overall Accuracy: {ensemble_metrics['accuracy']:.4f}")

## Feature Importance

Analyzing which features (Graph matching, Antonyms, Vectors, etc.) contribute most to the model's decisions.


In [ ]:
svd_names = [f'svd_vec_{i}' for i in range(30)]
feature_names = LOGICAL_FEATURE_NAMES + svd_names

importances = global_model.feature_importances_
indices = np.argsort(importances)[::-1]
for i in range(20):
    print(f'{i+1}. {feature_names[indices[i]]}: {importances[indices[i]]:.4f}')

plt.figure(figsize=(10, 8))
plt.barh(range(20), importances[indices[:20]][::-1], align='center')
plt.yticks(range(20), [feature_names[i] for i in indices[:20]][::-1])
plt.title('Top 20 Features (v5: Graph Kernels + Interacted Logical)')
plt.show()

## Error Analysis

Visualizing the confusion matrix and examining specific examples where the model succeeds or fails.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# 1. Confusion Matrix
cm = confusion_matrix(y_val, best_preds)
plt.figure(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['entailment', 'neutral', 'contradiction'])
disp.plot(cmap=plt.cm.Blues, ax=plt.gca())
plt.title('Confusion Matrix')
plt.show()

# 2. Example Analysis
labels_map = {0: 'entailment', 1: 'neutral', 2: 'contradiction'}
analysis_df = val_df_ordered.copy()
analysis_df['pred'] = best_preds

print("\n--- Detailed Prediction Examples ---")
for label_idx, label_name in labels_map.items():
    print(f"\n{'='*20} {label_name.upper()} {'='*20}")
    
    # Correct examples
    correct = analysis_df[(analysis_df['label'] == label_idx) & (analysis_df['pred'] == label_idx)].head(2)
    if not correct.empty:
        print(f"\n[CORRECTLY classified as {label_name.upper()}]")
        for _, row in correct.iterrows():
            print(f"P: {row['premise']}")
            print(f"H: {row['hypothesis']}")
            print("-")
            
    # Incorrect examples
    incorrect = analysis_df[(analysis_df['label'] == label_idx) & (analysis_df['pred'] != label_idx)].head(2)
    if not incorrect.empty:
        print(f"\n[INCORRECTLY classified (Should be {label_name.upper()})]")
        for _, row in incorrect.iterrows():
            print(f"P: {row['premise']}")
            print(f"H: {row['hypothesis']}")
            print(f"PREDICTED: {labels_map[row['pred']].upper()}")
            print("-")


## Playground

Use this cell to test the model with your own sentences.

In [ ]:
def predict_nli(premise, hypothesis, lang_abv='en', mode='global'):
    model_name = LANG_MAP.get(lang_abv, {'model': 'xx_sent_ud_sm'})['model']
    nlp_local = spacy.load(model_name)
    
    # 1. Extract raw features
    X_raw = extract_features(nlp_local, [premise], [hypothesis], lang_abv=lang_abv)
    X_vecs = X_raw[:, 0:N_VEC_TOTAL]
    X_logical = X_raw[:, N_VEC_TOTAL:]
    
    # 2. Transform using the specific transformer for this language
    transformer = lang_transformers.get(lang_abv)
    if transformer:
        X_final = transformer.transform(X_vecs, X_logical)
    else:
        X_final = np.hstack([X_logical, np.zeros((1, 30))])
    
    # 3. Predict
    if mode == 'global':
        pred_idx = global_model.predict(X_final)[0]
        probs = global_model.predict_proba(X_final)[0]
    else:
        # Use ensemble if model exists for this lang
        model = ensemble.models.get(lang_abv)
        if model:
            pred_idx = model.predict(X_final)[0]
            probs = model.predict_proba(X_final)[0]
        else:
            return "No model for this language", None
    
    res_labels = {0: 'entailment', 1: 'neutral', 2: 'contradiction'}
    label = res_labels[pred_idx]
    return label, probs

# --- TEST YOUR SENTENCES HERE ---
your_premise = "There is fire in forest."
your_hypothesis = "There isn't smoke in forest."
your_lang = "en"

label, probs = predict_nli(your_premise, your_hypothesis, lang_abv=your_lang, mode='ensemble')

print(f"Premise: {your_premise}")
print(f"Hypothesis: {your_hypothesis}")
print(f"\nPREDICTION: {label.upper()}")
print(f"Probabilities: Entailment: {probs[0]:.4f}, Neutral: {probs[1]:.4f}, Contradiction: {probs[2]:.4f}")

## Conclusion

The multilingual spaCy-based feature extraction pipeline combined with optimized XGBoost training provides a robust baseline for NLI across 15 languages. Key strategies include per-language model loading, multilingual WordNet lookups, and SVD-based vector compression.